In [1]:
# Imports and setup
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add workspace root to Python path for imports
workspace_root = Path('/workspace-hku/hku-datawork')
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

import cudf
import cupy as cp
RAPIDS_AVAILABLE = True
print("✓ RAPIDS available - using GPU acceleration")

import pandas as pd
import numpy as np
import logging
from tqdm import tqdm

# Import cointegration modules
from hypothesis_testing.cointegration import (
    load_price_data,
    generate_baskets_clustering,
    test_basket_cointegration,
    test_persistence_rolling,
    compute_max_zscore,
    optimize_lookback_window,
    optimize_basket_size,
    plot_spread_analysis,
    plot_lookback_optimization,
    print_summary_statistics
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


DATA_DIR = Path("../../../hku-data/test_data")


✓ RAPIDS available - using GPU acceleration


In [2]:
# Phase 1: Load price data
price_data = load_price_data(DATA_DIR, years_back=1.0)
symbols = [col.replace('_close', '') for col in price_data.columns]
print(f"\n✓ Data loaded: {len(symbols)} symbols, {len(price_data)} time points")

2025-10-31 10:30:12,912 - INFO - Loading data from 116 CSV files...
Loading CSVs: 100%|██████████| 116/116 [00:16<00:00,  7.01it/s]
2025-10-31 10:30:29,456 - INFO - Aligning timestamps across all symbols...
2025-10-31 10:30:29,963 - INFO - Loaded 30337 timestamps for 106 symbols
2025-10-31 10:30:29,965 - INFO - Date range: 2024-10-31 10:45:00+00:00 to 2025-09-12 10:45:00+00:00



✓ Data loaded: 106 symbols, 30337 time points


In [ ]:
# Phase 2 & 3: Generate baskets and test initial cointegration
logger.info("Generating candidate baskets...")
candidate_baskets = generate_baskets_clustering(price_data, n_clusters=8, 
                                                 min_basket_size=2, max_basket_size=6)
logger.info(f"Generated {len(candidate_baskets)} candidate baskets")


2025-10-31 10:30:29,992 - INFO - Generating candidate baskets...


In [ ]:

# Test initial cointegration
logger.info("Testing initial cointegration on full 1-year window...")
cointegrated_baskets = []
for basket in tqdm(candidate_baskets[:1000], desc="Testing baskets"):  # Limit for speed
    result = test_basket_cointegration(price_data, basket)
    if result:
        cointegrated_baskets.append(result)

logger.info(f"✓ Found {len(cointegrated_baskets)} cointegrated baskets")

In [ ]:
# Phase 4 & 5: Test persistence and spread constraints
logger.info("Testing persistence and spread constraints...")
valid_baskets = []

for basket_result in tqdm(cointegrated_baskets, desc="Persistence checks"):
    log_prices = basket_result['log_prices']
    eigenvector = basket_result['eigenvector']
    spread = basket_result['spread']
    
    # Test persistence
    persistence = test_persistence_rolling(log_prices, eigenvector, window_days=90, step_days=30)
    
    # Check max z-score
    max_z = compute_max_zscore(spread, lookback_days=90)
    
    # Size-dependent thresholds
    basket_size = len(basket_result['basket'])
    persistence_threshold = 0.8 - 0.1 * max(0, basket_size - 2)  # 0.8 for size 2, 0.7 for size 3, etc.
    z_threshold = 3.0 + 0.5 * max(0, basket_size - 2)  # 3.0 for size 2, 3.5 for size 3, etc.
    
    # Filter with size-dependent thresholds
    if persistence > persistence_threshold and max_z < z_threshold:
        basket_result['persistence'] = persistence
        basket_result['max_zscore'] = max_z
        basket_result['persistence_threshold'] = persistence_threshold
        basket_result['z_threshold'] = z_threshold
        valid_baskets.append(basket_result)

logger.info(f"✓ {len(valid_baskets)} baskets passed persistence and spread constraints")


2025-10-31 10:10:29,913 - INFO - Testing persistence and spread constraints...
Persistence checks: 100%|██████████| 310/310 [06:56<00:00,  1.34s/it]
2025-10-31 10:17:26,821 - INFO - ✓ 2 baskets passed persistence and spread constraints


In [ ]:
# Phase 6: Optimize lookback windows
logger.info("Optimizing lookback windows...")
optimized_baskets = []

for basket_result in tqdm(valid_baskets, desc="Window optimization"):
    log_prices = basket_result['log_prices']
    eigenvector = basket_result['eigenvector']
    spread = basket_result['spread']
    basket_size = len(basket_result['basket'])
    
    opt_result = optimize_lookback_window(log_prices, eigenvector, spread, basket_size=basket_size)
    
    if opt_result['optimal_window']:
        basket_result['optimal_lookback'] = opt_result['optimal_window']
        basket_result['lookback_metrics'] = opt_result['optimal_metrics']
        basket_result['lookback_results'] = opt_result['all_results']
        optimized_baskets.append(basket_result)

logger.info(f"✓ Optimized {len(optimized_baskets)} baskets with valid lookback windows")


2025-10-31 10:18:58,898 - INFO - Optimizing lookback windows...
Window optimization: 100%|██████████| 2/2 [00:16<00:00,  8.42s/it]
2025-10-31 10:19:15,752 - INFO - ✓ Optimized 2 baskets with valid lookback windows


In [ ]:
# Phase 7: Optimize basket size and select best
opt_results = optimize_basket_size(optimized_baskets)
best_basket = opt_results['best_overall']
baskets_by_size = opt_results['best_by_size']

logger.info(f"\n✓ Best baskets by size:")
for size in sorted(baskets_by_size.keys()):
    best = baskets_by_size[size]
    logger.info(f"  Size {size}: {best['basket']} (score: {best['final_score']:.3f}, "
                f"persistence: {best['persistence']:.3f}, max_z: {best['max_zscore']:.2f})")

if best_basket:
    logger.info(f"\n✓ Best overall basket: {best_basket['basket']}")
    logger.info(f"  Score: {best_basket['final_score']:.3f}")
    logger.info(f"  Persistence: {best_basket['persistence']:.3f}")
    logger.info(f"  Max z-score: {best_basket['max_zscore']:.2f}")
    logger.info(f"  Optimal lookback: {best_basket['optimal_lookback']} days")
    logger.info(f"  Johansen p-value: {best_basket['johansen_result']['p_value']:.4f}")
else:
    logger.warning("No valid baskets found!")

2025-10-31 10:20:11,359 - INFO - 
✓ Best baskets by size:
2025-10-31 10:20:11,363 - INFO -   Size 2: ['THETA-USDT-PERP', 'ONDO-USDT-PERP'] (score: 0.061, persistence: 0.875, max_z: 2.79)
2025-10-31 10:20:11,365 - INFO - 
✓ Best overall basket: ['THETA-USDT-PERP', 'ONDO-USDT-PERP']
2025-10-31 10:20:11,367 - INFO -   Score: 0.061
2025-10-31 10:20:11,369 - INFO -   Persistence: 0.875
2025-10-31 10:20:11,370 - INFO -   Max z-score: 2.79
2025-10-31 10:20:11,372 - INFO -   Optimal lookback: 180 days
2025-10-31 10:20:11,374 - INFO -   Johansen p-value: 0.0004


In [ ]:
# Phase 8: Visualization and reporting
if best_basket:
    plot_spread_analysis(best_basket, price_data)
    plot_lookback_optimization(best_basket)
    print_summary_statistics(best_basket)
else:
    print("No valid baskets found. Consider:")
    print("1. Relaxing persistence threshold (< 0.8)")
    print("2. Increasing max z-score limit (> 3.0)")
    print("3. Testing more basket combinations")


SUMMARY STATISTICS
Best Basket: THETA-USDT-PERP, ONDO-USDT-PERP
Basket Size: 2

Cointegration Test:
  Trace Statistic: 15.6095
  Critical Value (5%): 12.3212
  P-value: 0.0004

Persistence Metrics:
  Persistence Ratio: 0.875

Spread Statistics:
  Mean: 0.057947
  Std: 0.329052
  Max Z-Score: 2.79

Optimal Parameters:
  Lookback Window: 180 days
  Final Score: 0.061

Cointegration Vector (normalized):
  THETA-USDT-PERP: 1.000000
  ONDO-USDT-PERP: -2.401660
